# 1 &middot; Signal processing: one heartbeat, step by step

This notebook takes **one recording** and follows it all the way to the
26 numbers the models actually see.

| Step | What happens |
|---|---|
| 1 | Load a 20-second ECG recording |
| 2 | Clean it and find the heartbeats |
| 3 | Cut one beat around each heartbeat |
| 4 | Build the wavelet map and its ridge lines |
| 5 | Summarise the ridge lines into 26 numbers |

Every calculation is a call into `pipeline.py`, so nothing here can drift away
from the paper. Figures 1 and 2 of the manuscript are the polished versions of
the plots below.

Next notebook: `ecg_02_identity_verification.ipynb` trains the verifiers on
these features.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# import pipeline.py from the repo root, whether this runs from notebook/ or /
REPO_ROOT = next(p.resolve() for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pipeline.py").exists())
sys.path.insert(0, str(REPO_ROOT))
import pipeline

cfg = pipeline.PipelineConfig()          # the exact settings reported in the paper
cfg.db_dir = str(REPO_ROOT / "ecgiddb")

print("wavelet     :", cfg.wavelet_name)
print("beat window :", cfg.pre_ms, "ms before to", cfg.post_ms, "ms after the R-peak")
print("frequencies :", cfg.n_freqs, "steps from", cfg.freq_min, "to", cfg.freq_max, "Hz")
from ecg_features import load_ecg_id_from_physionet
if not load_ecg_id_from_physionet(cfg):
    raise RuntimeError("Download ECG-ID from PhysioNet before continuing")


## Steps 1-3 &middot; Load, clean, cut the beats

`pipeline.load_person` runs the same chain as the full experiment: read the
recording, remove baseline drift and muscle noise, find the R-peaks (the tall
spikes), cut a window around each one, and summarise every beat.

We unpack it here so each step is visible.

In [ ]:
p1 = pipeline.load_person(1, cfg)        # Person_01, rec_1

print(f"recording : {len(p1['signal'])} samples at {p1['fs']} Hz "
      f"= {len(p1['signal']) / p1['fs']:.1f} seconds")
print(f"heartbeats: {len(p1['r_peaks'])} R-peaks -> {len(p1['beats'])} usable beats")
print(f"one beat  : {p1['beats'].shape[1]} samples "
      f"({cfg.pre_ms + cfg.post_ms} ms)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6),
                         gridspec_kw={"width_ratios": [2, 1]})

# left: the cleaned recording with the detected heartbeats
seconds = np.arange(len(p1["cleaned"])) / p1["fs"]
window = seconds < 6                                  # first 6 s, so it stays readable
shown = p1["r_peaks"][p1["r_peaks"] < window.sum()]
axes[0].plot(seconds[window], p1["cleaned"][window], lw=0.8)
axes[0].scatter(shown / p1["fs"], p1["cleaned"][shown], s=25, color="crimson",
                zorder=3, label="R-peak")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Amplitude")
axes[0].set_title("Steps 1-2: cleaned recording, heartbeats found")
axes[0].legend(); axes[0].grid(alpha=0.3)

# right: the beats cut out of it, overlaid
for beat in p1["beats"][:8]:
    axes[1].plot(p1["beat_time"], (beat - beat.mean()) / (beat.std() + 1e-8),
                 lw=1, alpha=0.7)
axes[1].axvline(0, ls="--", color="crimson", label="R-peak")
axes[1].set_xlabel("Time from R-peak (s)"); axes[1].set_ylabel("Standardised")
axes[1].set_title("Step 3: eight beats, overlaid")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()

The right-hand plot is the property the whole method rests on: **one person's
beats repeat closely**. That is what makes it possible to learn an accept/reject
boundary from a single 20-second enrolment.

The window is longer after the R-peak (450 ms) than before it (250 ms), because
the T wave arrives after the peak.

## Step 4 &middot; The wavelet map and its ridge lines

The wavelet map shows *when* each frequency appears inside the beat: the QRS
spike is broad in frequency and narrow in time, the P and T waves are slow.

The **ridge lines** are just the local maxima of that map, followed from coarse
to fine scale. They trace the sharpest features of the beat, and they are what
the 26 numbers summarise.

In [ ]:
import pywt

power = np.abs(pywt.cwt(p1["beats"][0], p1["scales"], cfg.wavelet_name,
                        sampling_period=1 / p1["fs"])[0]) ** 2
lines = pipeline.compute_wtmm_lines(power, p1["scales"])

print(f"map shape  : {power.shape}  (frequencies x samples)")
print(f"ridge lines: {len(lines)} found in this beat")

In [ ]:
beat_seconds = np.arange(p1["beats"][0].size) / p1["fs"]
mesh_t, mesh_f = np.meshgrid(beat_seconds, p1["freqs"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, draw_lines, title in [(axes[0], False, "wavelet map"),
                              (axes[1], True, "same map + ridge lines")]:
    ax.pcolormesh(mesh_t, mesh_f, power, shading="auto", cmap="viridis")
    if draw_lines:
        for line in lines:
            ax.plot([t / p1["fs"] for _, t in line],
                    [p1["freqs"][s] for s, _ in line],
                    color="white", lw=0.8, alpha=0.9)
    ax.set_yscale("log"); ax.set_ylim(cfg.cut_freq, p1["freqs"].max())
    ax.set_xlabel("Time (s)"); ax.set_title(f"Step 4: {title}")
axes[0].set_ylabel("Frequency (Hz)")
plt.tight_layout()

## Step 5 &middot; The 26 numbers

The ridge lines become one short vector per beat:

- `total_lines` &mdash; how many ridge lines the beat has, and then
- for each of **5 frequency bands**: how many lines cross it, and the mean and
  spread of their frequency and of their power.

That is 1 + 5&times;5 = **26 values**, and it is everything the models receive.

In [ ]:
table = pd.DataFrame(p1["wtmm"], columns=pipeline.feature_names(cfg))

print("one row per beat, one column per feature:", table.shape)
table.iloc[:5, :7].round(3)

### Does it carry identity?

A quick check before the next notebook: the average feature vector of two
people, on the same axes. If the 26 numbers held no identity information the two
lines would sit on top of each other.

In [ ]:
p2 = pipeline.load_person(2, cfg)

# standardise both with one shared scale, so the comparison is readable
both = np.vstack([p1["wtmm"], p2["wtmm"]])
centre, spread = both.mean(0), both.std(0) + 1e-8

plt.figure(figsize=(12, 3.5))
plt.plot(pipeline.feature_names(cfg), (p1["wtmm"].mean(0) - centre) / spread,
         "o-", label="Person 01")
plt.plot(pipeline.feature_names(cfg), (p2["wtmm"].mean(0) - centre) / spread,
         "s-", label="Person 02")
plt.xticks(rotation=90, fontsize=7)
plt.ylabel("Standardised mean")
plt.title("Average feature vector of two people")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

## Take away

- A heartbeat becomes **26 numbers**, computed only from the ridge lines of its
  wavelet map.
- The overlays show variation among the selected beats.
- Differences between two people illustrate feature variation, but do not
  establish verification performance. The next notebook tests accept/reject decisions.

We call this representation *WTMM-inspired*: it summarises modulus maxima but
does not estimate a multifractal spectrum, and the paper makes no multifractal
claim.